# IMDb Movie Data — Validation & Database Exploration

This notebook validates and explores the processed movie dataset stored in SQL Server.

The dataset was collected from IMDb using Selenium and BeautifulSoup, processed and normalized with Python, and loaded into a relational SQL Server database.

### Pipeline

IMDb → Selenium + BeautifulSoup → Raw JSON → Pandas → Data Cleaning & Normalization → SQL Server

### Objectives

- Verify that the processed data was loaded correctly into SQL Server
- Validate data quality and consistency
- Check relationships between normalized tables
- Validate primary and foreign key relationships
- Test important data constraints
- Execute SQL queries against the final database
- Confirm that the dataset is ready for future data and AI/ML applications

In [1]:
import pandas as pd
import pyodbc

## SQL Server Connection

The processed dataset is stored in SQL Server using a normalized relational database design.

In [3]:
server = r'DESKTOP-BVMNIQD\MSSQLSERVEREXPRE'
database = 'movie_intelligence_db'
driver = 'ODBC Driver 17 for SQL Server'

conn = f"mssql+pyodbc://@{server}/{database}?driver={driver}&trusted_connection=yes"

print("Connected successfully!")

Connected successfully!


## Load Database Tables

The database contains normalized tables for movies, directors, actors, genres, and their many-to-many relationships.

In [4]:
movies = pd.read_sql("SELECT * FROM Movies", conn)
directors = pd.read_sql("SELECT * FROM Directors", conn)
actors = pd.read_sql("SELECT * FROM Actors", conn)
genres = pd.read_sql("SELECT * FROM Genres", conn)
movie_actors = pd.read_sql("SELECT * FROM Movie_Actors", conn)
movie_genres = pd.read_sql("SELECT * FROM Movie_Genres", conn)

In [5]:
tables = {
    "Movies": movies,
    "Directors": directors,
    "Actors": actors,
    "Genres": genres,
    "Movie_Actors": movie_actors,
    "Movie_Genres": movie_genres
}

for table_name, df in tables.items():
    print(f"{table_name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

Movies: 250 rows × 7 columns
Directors: 167 rows × 2 columns
Actors: 3,871 rows × 2 columns
Genres: 20 rows × 2 columns
Movie_Actors: 4,321 rows × 2 columns
Movie_Genres: 607 rows × 2 columns


## Database Structure Overview

The database follows a normalized relational structure.

### Main Entity Tables

- `Movies`
- `Directors`
- `Actors`
- `Genres`

### Relationship Tables

- `Movie_Actors`
- `Movie_Genres`

The relationship tables represent many-to-many relationships between movies and actors/genres.

In [6]:
for table_name, df in tables.items():
    print(f"\n--- {table_name} ---")
    print(df.columns.tolist())


--- Movies ---
['movie_id', 'director_id', 'title', 'movie_year', 'rating', 'votes', 'runtime_minutes']

--- Directors ---
['director_id', 'dir_name']

--- Actors ---
['actor_id', 'act_name']

--- Genres ---
['genre_id', 'gen_name']

--- Movie_Actors ---
['movie_id', 'actor_id']

--- Movie_Genres ---
['movie_id', 'genre_id']


## Sample Records

Inspect a small sample from each table to verify that the data structure and loaded values are correct.

In [7]:
for table_name, df in tables.items():
    print(f"\n--- {table_name} ---")
    display(df.head(3))


--- Movies ---


,movie_id,director_id,title,movie_year,rating,votes,runtime_minutes
0,1,52,The Godfather,1972,9.2,2300000,175
1,2,137,12 Angry Men,1957,9.0,1000000,96
2,3,143,Schindler's List,1993,9.0,1600000,196



--- Directors ---


,director_id,dir_name
0,1,Aamir Khan
1,2,Abbas Kiarostami
2,3,Aditya Chopra



--- Actors ---


...

,genre_id,gen_name
0,1,Action
1,2,Adventure
2,3,Animation



--- Movie_Actors ---


,movie_id,actor_id
0,1,7
1,1,61
2,1,62



--- Movie_Genres ---


,movie_id,genre_id
0,1,6
1,1,7
2,2,6


## Row Count Validation

Row counts are checked to confirm that records were loaded into the expected tables and that no table is unexpectedly empty.

In [10]:
for table_name, df in tables.items():
    status = "PASS" if len(df) > 0 else "FAIL"
    print(f"{table_name}: {len(df):,} rows → {status}")

Movies: 250 rows → PASS
Directors: 167 rows → PASS
Actors: 3,871 rows → PASS
Genres: 20 rows → PASS
Movie_Actors: 4,321 rows → PASS
Movie_Genres: 607 rows → PASS


In [11]:
empty_tables = [
    table_name
    for table_name, df in tables.items()
    if df.empty
]

print("Empty tables:", empty_tables)

Empty tables: []


## Missing Value Validation

Check for missing values across the loaded tables.

Missing values can indicate incomplete scraping, transformation issues, or unexpected database records.

In [12]:
for table_name, df in tables.items():
    print(f"\n--- {table_name} ---")
    display(df.isnull().sum())


--- Movies ---


movie_id           0
director_id        0
title              0
movie_year         0
rating             0
votes              0
runtime_minutes    0
dtype: int64


--- Directors ---


director_id    0
dir_name       0
dtype: int64


--- Actors ---


...

genre_id    0
gen_name    0
dtype: int64


--- Movie_Actors ---


movie_id    0
actor_id    0
dtype: int64


--- Movie_Genres ---


movie_id    0
genre_id    0
dtype: int64

In [13]:
null_summary = []

for table_name, df in tables.items():
    null_summary.append({
        "table": table_name,
        "total_nulls": int(df.isnull().sum().sum())
    })

null_summary = pd.DataFrame(null_summary)

display(null_summary)

,table,total_nulls
0,Movies,0
1,Directors,0
2,Actors,0
3,Genres,0
4,Movie_Actors,0
5,Movie_Genres,0


## Duplicate Validation

Check for duplicate records in the loaded tables.

Duplicate records can affect relationships and lead to incorrect downstream results.

In [14]:
for table_name, df in tables.items():
    duplicates = df.duplicated().sum()
    print(f"{table_name}: {duplicates:,} duplicate rows")

Movies: 0 duplicate rows
Directors: 0 duplicate rows
Actors: 0 duplicate rows
Genres: 0 duplicate rows
Movie_Actors: 0 duplicate rows
Movie_Genres: 0 duplicate rows


## Relationship Validation

The relationship tables should not contain duplicate movie-to-actor or movie-to-genre relationships.

In [15]:
movie_actor_duplicates = movie_actors.duplicated(
    subset=["movie_id", "actor_id"]
).sum()

movie_genre_duplicates = movie_genres.duplicated(
    subset=["movie_id", "genre_id"]
).sum()

print("Duplicate Movie-Actor relationships:", movie_actor_duplicates)
print("Duplicate Movie-Genre relationships:", movie_genre_duplicates)

Duplicate Movie-Actor relationships: 0
Duplicate Movie-Genre relationships: 0


## Movie Data Validation

Validate important constraints for movie-level attributes.

In [16]:
invalid_ratings = movies[
    (movies["rating"] < 0) |
    (movies["rating"] > 10)
]

print("Invalid ratings:", len(invalid_ratings))

Invalid ratings: 0


In [17]:
invalid_runtime = movies[
    movies["runtime_minutes"] <= 0
]

print("Invalid runtime values:", len(invalid_runtime))

Invalid runtime values: 0


In [18]:
invalid_votes = movies[
    movies["votes"] < 0
]

print("Invalid vote counts:", len(invalid_votes))

Invalid vote counts: 0


In [19]:
invalid_years = movies[
    (movies["movie_year"] < 1888) |
    (movies["movie_year"] > pd.Timestamp.now().year)
]

print("Invalid years:", len(invalid_years))

Invalid years: 0


## Foreign Key Validation

Verify that every `director_id` referenced by the Movies table exists in the Directors table.

In [20]:
invalid_directors = movies[
    ~movies["director_id"].isin(directors["director_id"])
]

print("Invalid director references:", len(invalid_directors))

Invalid director references: 0


In [21]:
invalid_movie_refs = movie_actors[
    ~movie_actors["movie_id"].isin(movies["movie_id"])
]

invalid_actor_refs = movie_actors[
    ~movie_actors["actor_id"].isin(actors["actor_id"])
]

print("Invalid movie references:", len(invalid_movie_refs))
print("Invalid actor references:", len(invalid_actor_refs))

Invalid movie references: 0
Invalid actor references: 0


In [22]:
invalid_movie_refs = movie_genres[
    ~movie_genres["movie_id"].isin(movies["movie_id"])
]

invalid_genre_refs = movie_genres[
    ~movie_genres["genre_id"].isin(genres["genre_id"])
]

print("Invalid movie references:", len(invalid_movie_refs))
print("Invalid genre references:", len(invalid_genre_refs))

Invalid movie references: 0
Invalid genre references: 0
